# Thích ứng Mô hình GPT qua Các Tác vụ NLP

## Cùng một mô hình — Định dạng đầu vào khác — Tác vụ khác

---

```
Input
  ↓
Prompt transformation
  ↓
GPT
  ↓
Prediction
```

**Mục tiêu:** Trình diễn cách một mô hình GPT duy nhất có thể thích ứng với nhiều tác vụ NLP (Question Answering, Natural Language Inference, Semantic Similarity) thông qua biến đổi định dạng đầu vào — mà không cần tinh chỉnh (fine-tuning) hay thay đổi kiến trúc mô hình.


In [ ]:
# @title Cài đặt & Thiết lập

import sys, os, warnings, logging
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

# Tự động cài đặt nếu đang chạy trên Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q transformers torch

print('=' * 50)
print('Đang tải thư viện...')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print('Đang tải GPT-2 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained('gpt2')
print('Đang tải GPT-2 model...')
model = AutoModelForCausalLM.from_pretrained('gpt2')

# ---- Xử lý Special Tokens (Option A) ----
# GPT-2 tokenizer mặc định không có <s>, $, <e>.
# Ta đăng ký chúng làm additional special tokens.
# Lưu ý quan trọng: embeddings của các token này được khởi tạo
# ngẫu nhiên và không phải là semantic tokens đã qua pretrain.
# Chúng chỉ đóng vai trò delimiter trực quan cho mục đích giáo dục.
print()
print('Đang đăng ký special tokens: <s>, $, <e>')

special_tokens_dict = {
    'additional_special_tokens': ['<s>', '$', '<e>']
}
num_added = tokenizer.add_special_tokens(special_tokens_dict)

# resize token embeddings vì từ điển đã thay đổi kích thước
model.resize_token_embeddings(len(tokenizer))

print(f'Đã thêm {num_added} special tokens.')
print(f'Tổng số tokens: {len(tokenizer)}')
print('Ready ✓')

# ---- GPT-3 / API (tuỳ chọn) ----
print()
try:
    import openai
    if 'OPENAI_API_KEY' in os.environ:
        print('GPT-3 API key found — optional comparison available')
    else:
        print('GPT-3 API key not found — continuing with GPT-2 only')
except ImportError:
    print('OpenAI library not installed — continuing with GPT-2 only')

print('=' * 50)

# ---- TaskPromptFormatter ----
class TaskPromptFormatter:
    """Định dạng input cho các tác vụ NLP bằng delimiter tokens.

    Sử dụng <s> (start), $ (delimiter), <e> (extraction)
    để biến đổi input thô thành prompt cho GPT model.
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.start_token = '<s>'
        self.delim_token = '$'
        self.extract_token = '<e>'

    def format_qa(self, context, question):
        text = (
            f'{self.start_token} '
            f'Context: {context} '
            f'Question: {question} '
            f'Answer: {self.extract_token}'
        )
        return self.tokenizer(text, return_tensors='pt')

    def format_entailment(self, premise, hypothesis):
        text = (
            f'{self.start_token} '
            f'Premise: {premise} '
            f'{self.delim_token} '
            f'Hypothesis: {hypothesis} '
            f'Relationship: {self.extract_token}'
        )
        return self.tokenizer(text, return_tensors='pt')

    def format_similarity(self, text1, text2):
        text = (
            f'{self.start_token} '
            f'Sentence A: {text1} '
            f'{self.delim_token} '
            f'Sentence B: {text2} '
            f'Similarity: {self.extract_token}'
        )
        return self.tokenizer(text, return_tensors='pt')

formatter = TaskPromptFormatter(tokenizer)
print('TaskPromptFormatter sẵn sàng.')
print('=' * 50)


---

### ⚠ Lưu ý quan trọng

**GPT-2 không được huấn luyện với instruction tuning.**

Chất lượng đầu ra (output quality) **không phải** mục tiêu của demo này.

Mục tiêu là **trình diễn cách biến đổi định dạng đầu vào** (prompt transformation) để cùng một mô hình có thể xử lý nhiều tác vụ khác nhau — mặc dù kết quả dự đoán có thể không chính xác tuyệt đối.

---


## Cách tiếp cận truyền thống vs. GPT

### Truyền thống:

```
Tác vụ A → Mô hình A (kiến trúc riêng, huấn luyện riêng)
Tác vụ B → Mô hình B (kiến trúc riêng, huấn luyện riêng)
Tác vụ C → Mô hình C (kiến trúc riêng, huấn luyện riêng)
```

### GPT:

```
Tác vụ A ─┐
Tác vụ B ─┼─→ Định dạng Prompt → Một mô hình GPT duy nhất
Tác vụ C ─┘
```

GPT dự đoán **token tiếp theo (next token prediction)**.

Mọi tác vụ đều được biến đổi thành bài toán sinh văn bản (text generation).


## Special Token Visualization

```
<s>  : start token — đánh dấu bắt đầu chuỗi
 $   : delimiter — phân tách giữa các đoạn văn bản
<e>  : extraction token — vị trí lấy kết quả dự đoán
```

Ví dụ với tác vụ Entailment:

```
Premise
  ↓
<s> Premise $ Hypothesis <e>
                       ↑
                   Kết quả dự đoán
```

> **Ghi chú kỹ thuật:** Embeddings của `<s>`, `$`, `<e>` được khởi tạo ngẫu nhiên và chưa qua pretrain. Chúng chỉ đóng vai trò delimiter trực quan, không mang ý nghĩa ngữ nghĩa.


---

## 1. Question Answering (QA)

---

### Input gốc:
```
Context:  The Eiffel Tower is in Paris.
Question: Where is the Eiffel Tower?
```

↓

### Prompt đã biến đổi:
```
<s> Context: The Eiffel Tower is in Paris.
    Question: Where is the Eiffel Tower?
    Answer: <e>
```

Mô hình sẽ dự đoán token tiếp theo sau `<e>`.

---


In [ ]:
def run_qa(context, question):
    print('=' * 50)
    print('QA Demo')
    print('=' * 50)
    print()
    print('Input gốc:')
    print(f'  Context:   {context}')
    print(f'  Question:  {question}')
    print()

    # Định dạng prompt
    inputs = formatter.format_qa(context, question)
    prompt_text = tokenizer.decode(inputs.input_ids[0])

    print('Prompt đã biến đổi:')
    print(f'  {prompt_text}')
    print()

    # Sinh câu trả lời
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            pad_token_id=tokenizer.eos_token_id
        )

    input_len = inputs.input_ids.shape[1]
    generated = outputs[0][input_len:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    print('Kết quả dự đoán:')
    print(f'  {answer}')
    print()
    print('=' * 50)
    return answer


qa_answer = run_qa(
    'The Eiffel Tower is in Paris.',
    'Where is the Eiffel Tower?'
)


---

## 2. Entailment (Natural Language Inference)

---

### Input gốc:
```
Premise:    A man is playing guitar.
Hypothesis: A person is making music.
```

↓

### Prompt đã biến đổi:
```
<s> Premise: A man is playing guitar.
    $
    Hypothesis: A person is making music.
    Relationship: <e>
```

Mô hình chọn một trong ba quan hệ:
- **Entailment** — giả thuyết được suy ra từ tiền đề
- **Neutral** — trung tính (không liên quan)
- **Contradiction** — mâu thuẫn với tiền đề

---


In [ ]:
def run_entailment(premise, hypothesis):
    print('=' * 50)
    print('Entailment Demo (NLI)')
    print('=' * 50)
    print()
    print('Input gốc:')
    print(f'  Premise:    {premise}')
    print(f'  Hypothesis: {hypothesis}')
    print()

    # Định dạng prompt
    inputs = formatter.format_entailment(premise, hypothesis)
    prompt_text = tokenizer.decode(inputs.input_ids[0])

    print('Prompt đã biến đổi:')
    print(f'  {prompt_text}')
    print()

    # Lấy logits tại vị trí cuối (sau <e>)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]

    # So sánh xác suất giữa các lựa chọn
    choices = ['Entailment', 'Neutral', 'Contradiction']
    log_probs = torch.log_softmax(logits, dim=-1)

    scores = {}
    for choice in choices:
        # Encode với khoảng trắng phía trước (token đầu sau <e>)
        tid = tokenizer.encode(' ' + choice, add_special_tokens=False)[0]
        scores[choice] = round(log_probs[tid].item(), 2)

    prediction = max(scores, key=scores.get)

    print('Điểm số (log-probability) từng lựa chọn:')
    for choice, score in scores.items():
        marker = '  ←' if choice == prediction else ''
        print(f'  {choice:15s}: {score:6.2f}{marker}')
    print()
    print(f'Dự đoán: {prediction}')
    print()
    print('=' * 50)
    return prediction, scores


ent_result, ent_scores = run_entailment(
    'A man is playing guitar.',
    'A person is making music.'
)


---

## 3. Semantic Similarity

---

### Input gốc:
```
Sentence A: I love dogs.
Sentence B: I like puppies.
```

↓

### Prompt đã biến đổi:
```
<s> Sentence A: I love dogs.
    $
    Sentence B: I like puppies.
    Similarity: <e>
```

Mô hình chọn một trong hai:
- **Similar** — tương tự
- **Not Similar** — không tương tự

> **Ghi chú:** Trong các nghiên cứu GPT ban đầu, thứ tự câu đôi khi được đảo ngược (Sentence B + Sentence A) vì thứ tự chuỗi có thể ảnh hưởng đến biểu diễn. Demo này chỉ sử dụng một thứ tự duy nhất.

---


In [ ]:
def run_similarity(text1, text2):
    print('=' * 50)
    print('Semantic Similarity Demo')
    print('=' * 50)
    print()
    print('Input gốc:')
    print(f'  Sentence A: {text1}')
    print(f'  Sentence B: {text2}')
    print()

    # Định dạng prompt
    inputs = formatter.format_similarity(text1, text2)
    prompt_text = tokenizer.decode(inputs.input_ids[0])

    print('Prompt đã biến đổi:')
    print(f'  {prompt_text}')
    print()

    # Lấy logits tại vị trí cuối (sau <e>)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]

    # So sánh xác suất giữa các lựa chọn
    choices = ['Similar', 'Not Similar']
    log_probs = torch.log_softmax(logits, dim=-1)

    scores = {}
    for choice in choices:
        # Encode với khoảng trắng phía trước (token đầu sau <e>)
        tid = tokenizer.encode(' ' + choice, add_special_tokens=False)[0]
        scores[choice] = round(log_probs[tid].item(), 2)

    prediction = max(scores, key=scores.get)

    print('Điểm số (log-probability) từng lựa chọn:')
    for choice, score in scores.items():
        marker = '  ←' if choice == prediction else ''
        print(f'  {choice:15s}: {score:6.2f}{marker}')
    print()
    print(f'Dự đoán: {prediction}')
    print()
    print('=' * 50)
    return prediction, scores


sim_result, sim_scores = run_similarity(
    'I love dogs.',
    'I like puppies.'
)


---

## Tổng kết

| Tác vụ | Input Changed | Model Changed |
|--------|:-------------:|:-------------:|
| QA (Question Answering) | ✓ | ✗ |
| NLI (Entailment) | ✓ | ✗ |
| Semantic Similarity | ✓ | ✗ |

### Kết luận

- **Cùng một mô hình** — GPT-2 duy nhất cho mọi tác vụ
- **Không fine-tuning** — không thay đổi trọng số mô hình
- **Chỉ thay đổi định dạng đầu vào** — mỗi tác vụ có prompt template riêng

> Mô hình ngôn ngữ dựa trên next-token-prediction có thể thích ứng với nhiều tác vụ NLP chỉ thông qua cách trình bày đầu vào (input transformation / prompt engineering).

---
